# AI JOB MARKET INTELLIGENCE
### An Exploratory Analysis of AI Jobs, Skills, Salaries and Career Trends

**Author:** Sagun Acharya  
**Project type:** Exploratory Data Analysis (EDA)  
**Execution:** Google Colab or Jupyter Notebook

This project uses a documented real-world job-posting dataset to investigate AI roles, salaries, experience requirements, work arrangements, technical skills, and career signals.

> **Goal:** Trusted Data → Clean EDA → Strong Questions → Meaningful Visualizations → Actionable Career Insights

This notebook intentionally avoids deep learning, predictive ML, dashboards, and unnecessary engineering. The emphasis is on interpretable exploratory analysis.

## 1. Dataset Source, Credibility & Scope

**Dataset:** AI Job Market Global 2026  
**Kaggle:** https://www.kaggle.com/datasets/atharvasoundankar/ai-job-market-global-2026  
**Original data sources:** Adzuna API and USAJobs API  
**Dataset license:** CC BY 4.0  
**Reported dataset size:** 5,773 job postings and 24 tracked technical skills

The dataset is a snapshot of job postings rather than a complete census of the global labor market. Salary values are not present for every posting, and skills are represented through the dataset's tracked/keyword-based skill fields.

### Automatic access
The notebook downloads the public Kaggle dataset at runtime with `kagglehub`. The dataset itself is intentionally **not committed to GitHub**.

## 2. Research Questions

### Job Market
1. Which AI job titles are most represented?
2. How is the market distributed across experience levels?
3. Which roles are more concentrated in entry, mid, senior, or executive positions?

### Salary
4. What does the observed salary distribution look like?
5. How does salary change with experience?
6. Which roles and locations have the highest observed median salaries?
7. How do salaries differ across work arrangements?

### Skills
8. Which technical skills are most frequently mentioned?
9. Which skills commonly appear together?
10. How do skill requirements differ between entry-level and senior roles?
11. Which skills are associated with higher observed salaries?

### Career Insight
12. Can a simple, transparent score summarize demand, salary, and entry accessibility without pretending to be a prediction model?

## 3. Imports, Paths & Visualization Style

In [ ]:
import sys
import subprocess
from pathlib import Path
from collections import Counter
from itertools import combinations
import re
import warnings

# Install dependencies when needed (works in Colab/Jupyter).
packages = {
    "kagglehub": "kagglehub>=0.3",
    "seaborn": "seaborn>=0.13",
}
for package_name, package_spec in packages.items():
    try:
        __import__(package_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_spec])

import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(context="notebook", style="whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

# Find a sensible project root in either Colab or a GitHub clone.
candidates = [
    Path.cwd(),
    Path("/content/AI-Job-Market-EDA"),
    Path("/content"),
]
PROJECT_ROOT = next((p for p in candidates if (p / "notebooks").exists()), Path.cwd())

VISUAL_DIR = PROJECT_ROOT / "visuals" / "important_charts"
VISUAL_DIR.mkdir(parents=True, exist_ok=True)

DATASET_SLUG = "atharvasoundankar/ai-job-market-global-2026"
DATASET_URL = "https://www.kaggle.com/datasets/atharvasoundankar/ai-job-market-global-2026"

def save_fig(filename):
    path = VISUAL_DIR / filename
    plt.tight_layout()
    plt.savefig(path, dpi=220, bbox_inches="tight")
    print(f"Saved → {path}")

print("Environment:", "Google Colab" if "google.colab" in sys.modules else "Local Jupyter")
print("Project root:", PROJECT_ROOT.resolve())
print("Chart directory:", VISUAL_DIR.resolve())

## 4. Automatic Dataset Download & Load

The loader downloads the dataset directly from Kaggle when no local CSV is available. It then selects the largest CSV in the downloaded package, avoiding a hard-coded filename dependency.

In [ ]:
# Prefer an existing project CSV if present; otherwise download automatically.
local_csvs = sorted((PROJECT_ROOT / "data").glob("*.csv")) if (PROJECT_ROOT / "data").exists() else []

if local_csvs:
    data_path = max(local_csvs, key=lambda p: p.stat().st_size)
    print("Using existing local CSV:", data_path)
else:
    print("Downloading dataset from Kaggle...")
    downloaded_path = Path(kagglehub.dataset_download(DATASET_SLUG))
    csv_files = list(downloaded_path.rglob("*.csv"))
    if not csv_files:
        raise FileNotFoundError("Kaggle download succeeded, but no CSV file was found.")
    data_path = max(csv_files, key=lambda p: p.stat().st_size)
    print("Downloaded dataset:", downloaded_path)

df = pd.read_csv(data_path)
print(f"Loaded dataset: {df.shape[0]:,} rows × {df.shape[1]:,} columns")
display(df.head())

## 5. Dataset Structure & Quality Audit

In [ ]:
print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]:,}")
print(f"Exact duplicate rows: {df.duplicated().sum():,}")

overview = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "non_null": df.notna().sum(),
    "missing": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique(dropna=True)
}).sort_values(["missing_pct", "unique_values"], ascending=[False, False])

display(overview)

## 6. Missing Values — Where Is Information Incomplete?

Missingness is treated as a data-quality characteristic rather than automatically filling values that were not supplied by employers.

In [ ]:
missing = (
    df.isna().mean().mul(100)
      .sort_values(ascending=False)
      .rename("missing_pct")
      .to_frame()
)

missing_plot = missing[missing["missing_pct"] > 0].head(15).sort_values("missing_pct")

plt.figure(figsize=(10, 6))
plt.barh(missing_plot.index, missing_plot["missing_pct"])
plt.xlabel("Missing values (%)")
plt.ylabel("Column")
plt.title("Where Job-Posting Data Is Missing")
plt.xlim(0, min(100, max(100, missing_plot["missing_pct"].max() * 1.08)))
save_fig("01_missing_values.png")
plt.show()
plt.close()

## 7. Cleaning & Feature Engineering

Cleaning is conservative: duplicates are removed, categorical codes are made readable, numeric fields are coerced, and existing salary values are used without inventing missing salaries.

In [ ]:
df = df.drop_duplicates().reset_index(drop=True).copy()

experience_map = {
    "EN": "Entry", "MI": "Mid", "SE": "Senior", "EX": "Executive",
    "Entry": "Entry", "Mid": "Mid", "Senior": "Senior", "Executive": "Executive"
}

if "experience_level" in df.columns:
    df["experience_label"] = df["experience_level"].map(experience_map).fillna(
        df["experience_level"].astype(str).str.title()
    )
else:
    df["experience_label"] = df["experience"].map(experience_map).fillna(
        df["experience"].astype(str).str.title()
    )

rr = pd.to_numeric(df["remote_ratio"], errors="coerce")
df["remote_label"] = pd.cut(
    rr, bins=[-1, 0, 50, 100],
    labels=["On-site", "Hybrid", "Remote"]
).astype("object")

for col in ["salary_usd", "years_experience", "job_description_length", "benefits_score", "remote_ratio"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

for date_col in ["date_posted", "posted_date", "collection_date"]:
    if date_col in df.columns:
        df[date_col] = pd.to_datetime(df[date_col], errors="coerce")

salary_df = df.dropna(subset=["salary_usd"]).copy()

required_columns = [
    "job_title", "experience_label", "remote_label",
    "salary_usd", "years_experience", "company_location",
    "required_skills"
]
missing_required = [c for c in required_columns if c not in df.columns]
if missing_required:
    raise ValueError(f"Expected dataset columns are missing: {missing_required}")

if salary_df.empty:
    raise ValueError("No usable salary observations were found; salary EDA cannot be performed.")

print(f"Cleaned dataset: {len(df):,} rows")
print(f"Salary-observed subset: {len(salary_df):,} rows ({len(salary_df)/len(df)*100:.1f}%)")
display(df[["job_title", "experience_label", "remote_label", "salary_usd", "years_experience"]].head())

## 8. Numeric Descriptive Statistics

In [ ]:
numeric_cols = ["salary_usd", "years_experience", "remote_ratio", "job_description_length", "benefits_score"]
numeric_summary = df[numeric_cols].describe().T
numeric_summary["missing"] = df[numeric_cols].isna().sum()
display(numeric_summary.round(2))

## 9. Most Common AI Roles

**Question:** Which job titles dominate this snapshot?

In [ ]:
role_counts = df["job_title"].value_counts().head(15)

plt.figure(figsize=(10, 7))
plt.barh(role_counts.index[::-1], role_counts.values[::-1])
plt.xlabel("Number of postings")
plt.ylabel("Job title")
plt.title("Top 15 AI Job Titles by Posting Count")
save_fig("02_top_roles.png")
plt.show()
plt.close()

## 10. Experience-Level Structure

A simple share chart shows how the observed market is distributed by experience requirement.

In [ ]:
experience_counts = df["experience_label"].value_counts().reindex(
    ["Entry", "Mid", "Senior", "Executive"]
).dropna()

plt.figure(figsize=(7, 7))
plt.pie(
    experience_counts.values,
    labels=experience_counts.index,
    autopct="%1.1f%%",
    startangle=90,
    wedgeprops={"linewidth": 1, "edgecolor": "white"}
)
plt.title("AI Job Postings by Experience Level")
plt.tight_layout()
save_fig("03_experience_share.png")
plt.show()
plt.close()

## 11. Role Mix Across Experience Levels

A 100% stacked bar chart makes differences in the composition of common roles easier to compare than raw counts alone.

In [ ]:
top_roles = df["job_title"].value_counts().head(10).index
role_exp = pd.crosstab(df["job_title"], df["experience_label"]).reindex(top_roles).fillna(0)
role_exp_pct = role_exp.div(role_exp.sum(axis=1), axis=0) * 100
role_exp_pct = role_exp_pct[["Entry", "Mid", "Senior", "Executive"]].fillna(0)

ax = role_exp_pct.iloc[::-1].plot(
    kind="barh", stacked=True, figsize=(11, 7)
)
ax.set_xlabel("Share of role's postings (%)")
ax.set_ylabel("Job title")
ax.set_title("Experience Composition of the 10 Most Common AI Roles")
ax.legend(title="Experience", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
save_fig("04_role_experience_mix.png")
plt.show()
plt.close()

## 12. Observed Salary Distribution & Outliers

Salary analysis uses only postings where salary is actually observed. Statistical outliers are retained because they may represent legitimate specialized or executive compensation.

In [ ]:
q1, q3 = salary_df["salary_usd"].quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outliers = salary_df[
    (salary_df["salary_usd"] < lower) |
    (salary_df["salary_usd"] > upper)
]

print(f"IQR bounds: ${lower:,.0f} to ${upper:,.0f}")
print(f"Potential statistical outliers: {len(outliers):,} ({len(outliers)/len(salary_df)*100:.1f}%)")

plt.figure(figsize=(10, 5))
sns.histplot(salary_df["salary_usd"], bins=35, kde=True)
plt.xlabel("Salary (USD)")
plt.ylabel("Job postings")
plt.title("Observed Salary Distribution")
save_fig("05_salary_distribution.png")
plt.show()
plt.close()

## 13. Salary by Experience Level

**Question:** Does observed compensation tend to rise across experience categories? This is descriptive, not causal.

In [ ]:
exp_order = ["Entry", "Mid", "Senior", "Executive"]
salary_exp = salary_df.dropna(subset=["experience_label"]).copy()

plt.figure(figsize=(9, 6))
sns.boxplot(
    data=salary_exp,
    x="experience_label",
    y="salary_usd",
    order=[x for x in exp_order if x in salary_exp["experience_label"].unique()]
)
plt.xlabel("Experience level")
plt.ylabel("Salary (USD)")
plt.title("Observed Salary Distribution by Experience Level")
plt.ylim(bottom=0)
save_fig("06_salary_by_experience.png")
plt.show()
plt.close()

## 14. Highest-Paying Roles

Roles are ranked by median salary and require at least five observed salaries to reduce the influence of extremely small samples.

In [ ]:
role_salary = (
    salary_df.groupby("job_title")["salary_usd"]
    .agg(postings="count", median_salary="median")
    .query("postings >= 5")
    .sort_values("median_salary", ascending=False)
    .head(15)
)

display(role_salary.style.format({"median_salary": "${:,.0f}"}))

plot_df = role_salary.sort_values("median_salary")
plt.figure(figsize=(10, 7))
plt.barh(plot_df.index, plot_df["median_salary"])
plt.xlabel("Median observed salary (USD)")
plt.ylabel("Job title")
plt.title("Top AI Roles by Median Observed Salary (≥5 salary observations)")
save_fig("07_high_paying_roles.png")
plt.show()
plt.close()

## 15. Salary by Location

Locations with too few salary observations are excluded from this ranking to avoid unstable medians.

In [ ]:
location_salary = (
    salary_df.groupby("company_location")["salary_usd"]
    .agg(postings="count", median_salary="median")
    .query("postings >= 8")
    .sort_values("median_salary", ascending=False)
    .head(15)
)

display(location_salary.style.format({"median_salary": "${:,.0f}"}))

plot_df = location_salary.sort_values("median_salary")
plt.figure(figsize=(10, 7))
plt.barh(plot_df.index, plot_df["median_salary"])
plt.xlabel("Median observed salary (USD)")
plt.ylabel("Location")
plt.title("Top Locations by Median Observed Salary (≥8 salary observations)")
save_fig("08_salary_by_location.png")
plt.show()
plt.close()

## 16. Work Arrangement Distribution & Salary

The first chart shows how postings are distributed across on-site, hybrid, and remote arrangements; the second compares observed salary distributions.

In [ ]:
remote_counts = df["remote_label"].value_counts().reindex(
    ["On-site", "Hybrid", "Remote"]
).dropna()

plt.figure(figsize=(8, 5))
plt.bar(remote_counts.index, remote_counts.values)
plt.xlabel("Work arrangement")
plt.ylabel("Job postings")
plt.title("AI Job Postings by Work Arrangement")
save_fig("09_work_arrangement_counts.png")
plt.show()
plt.close()

remote_salary = salary_df.dropna(subset=["remote_label"]).copy()
plt.figure(figsize=(8, 6))
sns.boxplot(
    data=remote_salary,
    x="remote_label",
    y="salary_usd",
    order=[x for x in ["On-site", "Hybrid", "Remote"] if x in remote_salary["remote_label"].unique()]
)
plt.xlabel("Work arrangement")
plt.ylabel("Salary (USD)")
plt.title("Observed Salary by Work Arrangement")
plt.ylim(bottom=0)
save_fig("10_salary_by_work_arrangement.png")
plt.show()
plt.close()

## 17. Skill Demand Ranking

Each posting contributes at most one count per normalized skill. The result measures **mention frequency**, not proficiency.

In [ ]:
def normalize_skill(s):
    s = str(s).strip()
    aliases = {
        "scikit-learn": "Scikit-learn", "sklearn": "Scikit-learn",
        "huggingface": "HuggingFace", "hugging face": "HuggingFace",
        "gcp": "GCP", "aws": "AWS", "azure": "Azure",
        "pytorch": "PyTorch", "tensorflow": "TensorFlow", "keras": "Keras",
        "python": "Python", "sql": "SQL", "r": "R", "openai": "OpenAI",
        "mlflow": "MLflow", "airflow": "Airflow",
        "computer vision": "Computer Vision", "fine-tuning": "Fine-tuning",
        "finetuning": "Fine-tuning", "transformers": "Transformers",
        "langchain": "LangChain", "kubernetes": "Kubernetes",
        "docker": "Docker", "spark": "Spark", "hadoop": "Hadoop",
        "nlp": "NLP", "rag": "RAG"
    }
    return aliases.get(s.lower(), s.title())

def parse_skill_set(raw):
    if pd.isna(raw):
        return set()
    return {
        normalize_skill(x)
        for x in re.split(r",|;|\||\n", str(raw))
        if str(x).strip()
    }

skill_sets = df["required_skills"].apply(parse_skill_set)
skill_counter = Counter(skill for skills in skill_sets for skill in skills)
skill_rank = pd.Series(skill_counter).sort_values(ascending=False)

skill_table = pd.DataFrame({
    "postings": skill_rank.astype(int),
    "share_of_postings_pct": (skill_rank / len(df) * 100).round(2)
})

display(skill_table.head(20))

top_skills = skill_table.head(15).sort_values("postings")
plt.figure(figsize=(10, 7))
plt.barh(top_skills.index, top_skills["share_of_postings_pct"])
plt.xlabel("Share of job postings mentioning skill (%)")
plt.ylabel("Skill")
plt.title("Top 15 Technical Skills by Job-Posting Mentions")
save_fig("11_skill_demand.png")
plt.show()
plt.close()

## 18. Skill Co-Occurrence — Common Skill Pairings 

This analysis ranks the most common **skill pairs** directly, making it easier to read as a practical learning-stack view.

In [ ]:
pair_counter = Counter()
for skills in skill_sets:
    pair_counter.update(combinations(sorted(skills), 2))

pair_df = pd.DataFrame(
    [(a, b, n) for (a, b), n in pair_counter.items()],
    columns=["skill_1", "skill_2", "cooccurrence"]
).sort_values("cooccurrence", ascending=False)

top_pairs = pair_df.head(12).copy()
top_pairs["pair"] = top_pairs["skill_1"] + " + " + top_pairs["skill_2"]
display(top_pairs[["pair", "cooccurrence"]])

plot_pairs = top_pairs.sort_values("cooccurrence")
plt.figure(figsize=(10, 7))
plt.barh(plot_pairs["pair"], plot_pairs["cooccurrence"])
plt.xlabel("Number of postings mentioning both skills")
plt.ylabel("Skill pair")
plt.title("Most Common Technical Skill Pairings")
save_fig("12_skill_pairings.png")
plt.show()
plt.close()

## 19. Entry vs. Senior Skill Demand 

The dumbbell-style chart compares the percentage of entry and senior postings mentioning each of the most common skills.

In [ ]:
comparison_skills = skill_table.head(12).index.tolist()
levels = ["Entry", "Senior"]

skill_level_rows = []
for level in levels:
    subset = df.loc[df["experience_label"] == level, "required_skills"].apply(parse_skill_set)
    n = len(subset)
    for skill in comparison_skills:
        mentions = sum(skill in skills for skills in subset)
        skill_level_rows.append([skill, level, mentions / n * 100])

level_skill = pd.DataFrame(
    skill_level_rows, columns=["skill", "experience", "share_pct"]
)
wide = level_skill.pivot(index="skill", columns="experience", values="share_pct").dropna()

plot_df = wide.sort_values("Senior")
fig, ax = plt.subplots(figsize=(10, 7))

y = np.arange(len(plot_df))
ax.hlines(y, plot_df["Entry"], plot_df["Senior"], linewidth=2)
ax.scatter(plot_df["Entry"], y, s=55, label="Entry")
ax.scatter(plot_df["Senior"], y, s=55, label="Senior")

ax.set_yticks(y)
ax.set_yticklabels(plot_df.index)
ax.set_xlabel("Share of postings mentioning skill (%)")
ax.set_ylabel("Skill")
ax.set_title("Entry vs. Senior Skill Mentions")
ax.legend()
plt.tight_layout()
save_fig("13_entry_vs_senior_skills.png")
plt.show()
plt.close()

## 20. Skill–Salary Association 

For common skills, compare the median salary of postings mentioning the skill with the overall salary median. **This is association, not causation and not a guaranteed salary premium.**

In [ ]:
overall_median = salary_df["salary_usd"].median()
skill_salary_rows = []

for skill in skill_rank.index:
    mask = salary_df["required_skills"].apply(lambda x: skill in parse_skill_set(x))
    sub = salary_df.loc[mask, "salary_usd"]
    if len(sub) >= 5:
        median = sub.median()
        skill_salary_rows.append([
            skill, len(sub), median, median - overall_median
        ])

skill_salary = pd.DataFrame(
    skill_salary_rows,
    columns=["skill", "salary_postings", "median_salary", "difference_vs_overall_median"]
).sort_values("difference_vs_overall_median", ascending=False)

display(
    skill_salary.head(15).style.format({
        "median_salary": "${:,.0f}",
        "difference_vs_overall_median": "${:,.0f}"
    })
)

assoc = pd.concat([
    skill_salary.head(7),
    skill_salary.tail(7)
]).drop_duplicates().sort_values("difference_vs_overall_median")

plt.figure(figsize=(10, 8))
plt.barh(assoc["skill"], assoc["difference_vs_overall_median"])
plt.axvline(0, linestyle="--", linewidth=1)
plt.xlabel("Median salary difference vs. overall salary median (USD)")
plt.ylabel("Skill")
plt.title("Skills with Higher/Lower Observed Median Salary Association")
save_fig("14_skill_salary_association.png")
plt.show()
plt.close()

## 21. Experience vs. Salary — Relationship View

A scatter plot shows the relationship between required years of experience and observed salary, with a linear trend line used only as a descriptive guide.

In [ ]:
scatter_df = salary_df.dropna(subset=["years_experience", "salary_usd"]).copy()

plt.figure(figsize=(10, 6))
sns.regplot(
    data=scatter_df,
    x="years_experience",
    y="salary_usd",
    scatter_kws={"alpha": 0.30, "s": 35},
    line_kws={"linewidth": 2}
)
plt.xlabel("Required years of experience")
plt.ylabel("Salary (USD)")
plt.title("Observed Salary vs. Required Years of Experience")
plt.ylim(bottom=0)
save_fig("15_salary_vs_experience.png")
plt.show()
plt.close()

## 22. Role Demand vs. Salary — Market Positioning

This combines two EDA dimensions without turning them into a prediction model: posting volume and observed median salary. Bubble size represents the number of salary observations.

In [ ]:
role_market = (
    salary_df.groupby("job_title")
    .agg(
        postings=("job_title", "size"),
        median_salary=("salary_usd", "median")
    )
)

role_market["total_postings"] = df["job_title"].value_counts()
role_market = role_market[
    (role_market["postings"] >= 5) &
    (role_market["total_postings"] >= 10)
].copy()

plt.figure(figsize=(10, 7))
plt.scatter(
    role_market["total_postings"],
    role_market["median_salary"],
    s=role_market["postings"] * 7,
    alpha=0.6
)

top_labels = role_market.nlargest(8, "total_postings")
for role, row in top_labels.iterrows():
    plt.annotate(
        role,
        (row["total_postings"], row["median_salary"]),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=9
    )

plt.xlabel("Total job postings for role")
plt.ylabel("Median observed salary (USD)")
plt.title("Role Demand vs. Median Observed Salary")
plt.ylim(bottom=0)
save_fig("16_role_demand_vs_salary.png")
plt.show()
plt.close()

## 23. Job Description Length by Experience

This is an additional content-intensity check: do higher-level job postings tend to contain longer descriptions?

In [ ]:
desc_df = df.dropna(subset=["job_description_length", "experience_label"]).copy()

plt.figure(figsize=(9, 6))
sns.boxplot(
    data=desc_df,
    x="experience_label",
    y="job_description_length",
    order=[x for x in exp_order if x in desc_df["experience_label"].unique()]
)
plt.xlabel("Experience level")
plt.ylabel("Job description length")
plt.title("Job Description Length by Experience Level")
plt.ylim(bottom=0)
save_fig("17_description_length_by_experience.png")
plt.show()
plt.close()

## 24. Correlation Analysis — Numeric Relationships

A correlation coefficient summarizes linear association between numeric variables. It is not evidence of causation. A **bar chart of pairwise correlations** is used so the direction and strength of each relationship can be read directly.

In [ ]:
cor_cols = ["salary_usd", "years_experience", "remote_ratio",
            "job_description_length", "benefits_score"]

corr = df[cor_cols].corr(numeric_only=True)
pairs = []
for a, b in combinations(cor_cols, 2):
    value = corr.loc[a, b]
    pairs.append([f"{a} ↔ {b}", value])

corr_pairs = pd.DataFrame(pairs, columns=["pair", "correlation"])
corr_pairs["abs_corr"] = corr_pairs["correlation"].abs()
corr_pairs = corr_pairs.sort_values("abs_corr").drop(columns="abs_corr")

display(corr_pairs.round(3))

plt.figure(figsize=(10, 7))
plt.barh(corr_pairs["pair"], corr_pairs["correlation"])
plt.axvline(0, linestyle="--", linewidth=1)
plt.xlabel("Pearson correlation")
plt.ylabel("Numeric variable pair")
plt.title("Pairwise Numeric Correlations")
save_fig("18_numeric_correlations.png")
plt.show()
plt.close()

## 25. Simple AI Career Opportunity Score 

This is an **exploratory ranking**, not a hiring model or salary predictor.

**Career Score = 40% Demand + 35% Salary + 25% Entry Accessibility**

- **Demand:** percentile rank of total postings for the role.
- **Salary:** percentile rank of median observed salary.
- **Entry accessibility:** inverse percentile of median required years of experience.

Roles require enough observations to keep the score reasonably stable.

In [ ]:
role_base = df.groupby("job_title").agg(
    postings=("job_title", "size"),
    median_years_experience=("years_experience", "median")
)

salary_role = salary_df.groupby("job_title").agg(
    median_salary=("salary_usd", "median"),
    salary_observed=("salary_usd", "count")
)

score_df = role_base.join(salary_role, how="inner")
score_df = score_df[
    (score_df["postings"] >= 10) &
    (score_df["salary_observed"] >= 5)
].dropna(subset=["median_salary", "median_years_experience"]).copy()

score_df["demand_pct"] = score_df["postings"].rank(pct=True) * 100
score_df["salary_pct"] = score_df["median_salary"].rank(pct=True) * 100
score_df["entry_accessibility_pct"] = (
    1 - score_df["median_years_experience"].rank(pct=True)
) * 100

score_df["career_score"] = (
    0.40 * score_df["demand_pct"] +
    0.35 * score_df["salary_pct"] +
    0.25 * score_df["entry_accessibility_pct"]
)

score_df = score_df.sort_values("career_score", ascending=False)

display(
    score_df.head(15)[[
        "postings", "salary_observed", "median_salary",
        "median_years_experience", "demand_pct",
        "salary_pct", "entry_accessibility_pct", "career_score"
    ]].style.format({
        "median_salary": "${:,.0f}",
        "median_years_experience": "{:.1f}",
        "demand_pct": "{:.1f}",
        "salary_pct": "{:.1f}",
        "entry_accessibility_pct": "{:.1f}",
        "career_score": "{:.1f}"
    })
)

plot_score = score_df.head(12).sort_values("career_score")

plt.figure(figsize=(10, 7))
plt.barh(plot_score.index, plot_score["career_score"])
plt.xlabel("Career Opportunity Score (0–100)")
plt.ylabel("Job title")
plt.title("Exploratory AI Career Opportunity Score")
plt.xlim(0, 100)
save_fig("19_career_opportunity_score.png")
plt.show()
plt.close()

## 26. Key Findings

The following statements are generated from the observed data rather than written as generic career advice.

In [ ]:
findings = []

top_role, top_role_n = role_counts.index[0], int(role_counts.iloc[0])
findings.append(
    f"The most represented job title is **{top_role}**, with **{top_role_n:,} postings**."
)

top_exp = df["experience_label"].value_counts()
findings.append(
    f"The largest experience category is **{top_exp.index[0]}**, representing "
    f"**{top_exp.iloc[0] / len(df) * 100:.1f}%** of postings."
)

findings.append(
    f"Salary is observed for **{len(salary_df) / len(df) * 100:.1f}%** of postings; "
    "salary conclusions therefore apply only to the disclosed-salary subset."
)

findings.append(
    f"**{skill_rank.index[0]}** is the most frequently mentioned tracked skill, "
    f"appearing in approximately **{skill_rank.iloc[0] / len(df) * 100:.1f}%** of postings."
)

exp_medians = salary_df.groupby("experience_label")["salary_usd"].median().dropna()
exp_medians = exp_medians.reindex([x for x in exp_order if x in exp_medians.index]).dropna()
findings.append(
    f"Median observed salary ranges from **${exp_medians.min():,.0f}** to "
    f"**${exp_medians.max():,.0f}** across represented experience levels."
)

highest_role = role_salary.index[0]
findings.append(
    f"Among roles meeting the minimum salary-observation threshold, **{highest_role}** "
    f"has the highest median observed salary at **${role_salary.iloc[0]['median_salary']:,.0f}**."
)

top_pair = pair_df.iloc[0]
findings.append(
    f"The most common tracked skill pairing is **{top_pair['skill_1']} + {top_pair['skill_2']}**, "
    f"appearing together in **{int(top_pair['cooccurrence']):,} postings**."
)

if not score_df.empty:
    findings.append(
        f"Under the exploratory Career Opportunity Score, **{score_df.index[0]}** ranks first; "
        "the score combines demand, observed salary, and entry accessibility."
    )

for i, finding in enumerate(findings[:8], 1):
    print(f"{i}. {finding}")

## 27. AI Career Takeaways

The data supports a **stack-building approach** rather than learning isolated tools.

1. Start with the most widely mentioned foundational skills in the dataset, especially those appearing frequently across many role types.
2. Add complementary tools that frequently co-occur with those foundational skills; this better reflects how requirements appear together in job postings.
3. Distinguish **entry-level demand from senior specialization**. Skills that are common in senior postings are not automatically the best first skills for a student.
4. Use salary-associated skills and high-paying roles as **signals for investigation**, not guarantees of compensation.
5. Consider role demand, salary, and experience requirements together rather than optimizing for only one variable.

These takeaways describe the patterns in this dataset snapshot and should not be interpreted as universal rules for the entire AI labor market.

## 28. Limitations & Responsible Interpretation

- **Snapshot bias:** the dataset represents a particular collection period and is not a historical time series.
- **Source coverage:** job postings collected through Adzuna and USAJobs do not represent every employer or every country equally.
- **Salary missingness:** many postings do not disclose salary, so salary results describe only the observed-salary subset.
- **Skill representation:** tracked skills are based on structured/keyword information rather than a test of actual skill proficiency.
- **Role naming:** job titles vary across employers, so title-level comparisons can split similar roles into separate categories.
- **Association ≠ causation:** relationships between salary, skills, experience, location, and remote work may be influenced by other factors.
- **Career Opportunity Score:** a transparent EDA index, not a prediction of hiring probability, career success, or future salary.

## Dataset Attribution

Soundankar, A. (2026). *AI Job Market Global 2026* [Dataset]. Kaggle.  
https://www.kaggle.com/datasets/atharvasoundankar/ai-job-market-global-2026

**Original collection sources:** Adzuna API and USAJobs API  
**Dataset license:** CC BY 4.0

The analysis uses the dataset for educational and exploratory purposes and retains attribution to the dataset creator and source.